# Causal activation patching

Patch head-1 output from one example into another.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]
    if (candidate / "results/line_breaks_results").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/06_layer1_mechanisms"))
from common import *
EXAMPLE_SEED=6001
ctx=load_context(EXAMPLE_SEED)
model,vocab,batch,cfg,device=ctx.model,ctx.vocab,ctx.batch,ctx.cfg,ctx.device
tokens=batch.tokens[0].detach().cpu().numpy()
print("checkpoint",ctx.checkpoint["step"],"seed",EXAMPLE_SEED,"line width",int(batch.line_lengths[0]))
print("newline positions",newline_positions(tokens,vocab))

/Users/rohanbuluswar/Desktop/Line Break Project/.venv_new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Building vocab...
('BOS', '_NEWLINE_')
checkpoint 10000 seed 6001 line width 89
newline positions [22, 44, 66, 87, 106, 129, 149]


In [2]:
src,tgt=load_context(6012),load_context(6013); Z,POST="blocks.1.attn.hook_z","blocks.1.hook_resid_post"
_,cs=src.model.run_with_cache(src.batch.tokens,prepend_bos=False,names_filter=[Z,POST]); _,ct=tgt.model.run_with_cache(tgt.batch.tokens,prepend_bos=False,names_filter=[POST])
sz=cs[Z][0,:,1].detach().clone(); sp=cs[POST][0].detach().cpu().numpy(); tp=ct[POST][0].detach().cpu().numpy(); capd={}
def patch(x,hook):y=x.clone();y[:,:,1,:]=sz.to(device=y.device,dtype=y.dtype);return y
def cap(x,hook):capd["post"]=x.detach().cpu();return x
tgt.model.run_with_hooks(tgt.batch.tokens,prepend_bos=False,fwd_hooks=[(Z,patch),(POST,cap)])
pp=capd["post"][0].numpy(); print("target clean to source MSE",float(np.mean((tp-sp)**2))); print("patched to source MSE",float(np.mean((pp-sp)**2)))

Building vocab...
('BOS', '_NEWLINE_')
Building vocab...
('BOS', '_NEWLINE_')
target clean to source MSE 8.836321830749512
patched to source MSE 4.884028434753418
